# Figure PS01 — Phage: the minus-strand label error and the shortcut it taught the model

**Status:** draft (Setonix integrator, agent-handoffs#24). Provisional ID; supplementary.

**Caption (draft).** Mean gain (5 grouped cross-validation folds; whiskers min–max) of each feature in the original 14-feature ranking model (31,200 genomes, 5,887,690 ORFs), trained on the corrected labels. 3Di entropy carries 0.67 of the gain; strand, frame and position together carry less than 0.02, so the model does not rely on where an ORF lies. All phage results in the paper use the corrected labels.

**Manuscript text:** Methods, phage labels; Results, phage section.

## Provenance

| item | value |
|---|---|
| plotting data | `../data/figPS01_label_audit.tsv.gz`, `../data/figPS01_feature_importance.tsv.gz`; sha256 in `../data/phage_provenance.json` |
| produced by | `manuscript/prep/phage_prepare_data.py` |
| upstream | `Validation/mutual_information/tables/label_legacy_vs_v2.tsv` (#108, `scripts/compare_labels.py`); `Validation/Inphared/tables/model_importance.tsv` (#110, `scripts/03_models.py`) |
| populations | the ranking ORF set (49 of 361 Inphared parts lack entropy JSON) — see `methods_provenance.md`. `figPS01_label_audit` (legacy vs corrected labels by strand) is kept for provenance but no longer plotted. |

## Data dictionary

| file | columns | meaning |
|---|---|---|
| `figPS01_label_audit` | `strand`, `contig_edge`, `legacy`, `v2`, `orfs` | ORF counts by label agreement |
| `figPS01_feature_importance` | `set` (`orig14`), `labels` (`json` = legacy, `relabelled` = corrected), `feature`, `gain_mean/min/max` | XGBoost gain over 5 folds |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import msfig

msfig.style()
SLUG = "figPS01_phage_label_correction"
C = msfig.PALETTE

In [ ]:
audit = msfig.data("figPS01_label_audit.tsv.gz")
imp = msfig.data("figPS01_feature_importance.tsv.gz")

# Label audit: kept as provenance for the Methods text (corrected matcher labels 1,000,839 minus-strand genes).
pos = {(s, lab): audit[(audit.strand == s) & audit[lab]].orfs.sum()
       for s in "+-" for lab in ("legacy", "v2")}
assert pos["-", "legacy"] == 3 and pos["-", "v2"] == 1_000_839
assert audit.orfs.sum() == 6_754_361

g = imp[(imp.set == "orig14") & (imp.labels == "relabelled")].set_index("feature")
assert len(g) == 14
assert round(g.loc["three_di_entropy", "gain_mean"], 2) == 0.67
assert g.gain_mean.idxmax() == "three_di_entropy"
assert g.loc[["strand_plus", "frame", "start", "end"], "gain_mean"].sum() < 0.02

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.2))
d = g.sort_values("gain_mean")
y = np.arange(len(d))
ax.barh(y, d.gain_mean, 0.62, color=C["blue"],
        xerr=[d.gain_mean - d.gain_min, d.gain_max - d.gain_mean],
        error_kw=dict(lw=0.6, capsize=1))
NAMES = {"three_di": "3Di", "twelve_state": "12-state", "dna": "DNA", "strand_plus": "strand (plus)"}


def name(f):
    for k, v in NAMES.items():
        f = f.replace(k, v)
    return f.replace("_", " ")


ax.set_yticks(y, [name(f) for f in d.index], fontsize=6.5)
ax.set_xlabel("feature gain (share of total)")
ax.set_xlim(0, 0.8)

fig.tight_layout()
msfig.save(fig, SLUG)